# 🎬 LTX-2 Video Generator — Free **Kaggle** UI

This notebook runs the LTX-2 prompt-to-video Gradio UI on Kaggle's **free T4 ×2 GPU**.

## Disk strategy

Kaggle has two writable volumes:

- `/kaggle/working` — only **~20 GB**, persisted between cells.
- `/tmp` — **~60-70 GB**, volatile (wiped if the session restarts), but plenty of room for the 30 GB of LTX-2 weights.

We download weights to `/tmp/hf_cache`. They survive across cells in the same session, which is all we need to keep the Gradio server running. (If your session restarts you'll have to re-download — that's the trade-off for using `/tmp`.)

## Steps

1. Open <https://www.kaggle.com/code> → **+ New Notebook**.
2. **File → Import notebook → URL** → paste:
   `https://raw.githubusercontent.com/debzody/LTX-2/main/kaggle_ltx2_ui.ipynb`
3. Right sidebar:
   - **Settings → Accelerator → GPU T4 ×2** (or **GPU P100**).
   - **Settings → Persistence → Files only**.
   - **Settings → Internet → On** (required to download weights).
   - **Add-ons → Secrets → New secret** `HF_TOKEN` = your HF read token.
4. **Run All**.
5. Cell 6 prints a public `https://*.gradio.live` URL — open it.

Phone-verify your Kaggle account first (Settings → Profile) or the GPU dropdown will be locked.

## 1. Sanity check

In [ ]:
!nvidia-smi || echo 'No GPU. Right-side panel: Accelerator > GPU T4 x2'
print('---'); !df -h /kaggle/working /tmp 2>/dev/null

## 2. Clone the repo and pull the UI files

In [ ]:
import os
os.chdir('/kaggle/working')
if not os.path.exists('LTX-2'):
    !git clone --depth 1 https://github.com/Lightricks/LTX-2.git
%cd /kaggle/working/LTX-2
FORK = 'https://raw.githubusercontent.com/debzody/LTX-2/main'
!curl -sSL {FORK}/app.py -o app.py
!curl -sSL {FORK}/app_pipeline.py -o app_pipeline.py
!ls -la app.py app_pipeline.py

## 3. Install LTX-2 + Gradio (~5 min)

In [ ]:
!pip install -q uv
!uv pip install --system -e packages/ltx-core -e packages/ltx-pipelines
!uv pip install --system 'gradio>=4.44'

## 4. Hugging Face login (Gemma is gated)

Best on Kaggle: store the token as a **Kaggle Secret** (Add-ons → Secrets → label `HF_TOKEN`).

Otherwise the cell will prompt you to paste it.

In [ ]:
import os
from huggingface_hub import login, whoami

token = os.environ.get('HF_TOKEN')
if not token:
    try:
        from kaggle_secrets import UserSecretsClient
        token = UserSecretsClient().get_secret('HF_TOKEN')
    except Exception as e:
        print('No Kaggle Secret named HF_TOKEN.', e)
if not token:
    from getpass import getpass
    token = getpass('HF token (hf_...): ').strip()
if not token:
    raise RuntimeError('No HF token. See cell 4 description above.')
login(token=token, add_to_git_credential=False)
os.environ['HF_TOKEN'] = token
print('Logged in as:', whoami()['name'])

## 5. Download model weights to `/tmp/hf_cache` (~30 GB)

We use `/tmp` because `/kaggle/working` only has ~20 GB. `/tmp` is volatile — wiped if your session restarts — but for one running session that's fine.

We download only the distilled checkpoint + Gemma — no upsampler — for the `one-stage` pipeline.

In [ ]:
import os, json, shutil
from huggingface_hub import hf_hub_download, snapshot_download
from huggingface_hub.utils import GatedRepoError

CACHE = '/tmp/hf_cache'
os.makedirs(CACHE, exist_ok=True)
os.environ['HF_HOME'] = CACHE
os.environ['HUGGINGFACE_HUB_CACHE'] = CACHE

free = shutil.disk_usage('/tmp').free / (1024**3)
print(f'Free disk on /tmp: {free:.1f} GB')
if free < 35:
    raise SystemExit(
        f'Need >=35 GB free on /tmp, got {free:.1f} GB. '
        'Restart the runtime: Run > Restart & clear all outputs.'
    )

print('Downloading distilled LTX-2 checkpoint (~22 GB)...')
ckpt = hf_hub_download(
    repo_id='Lightricks/LTX-2.3',
    filename='ltx-2.3-22b-distilled-1.1.safetensors',
    cache_dir=CACHE,
)
print('  ', ckpt)

print('Downloading Gemma-3 text encoder (~7 GB)...')
try:
    gemma = snapshot_download(
        repo_id='google/gemma-3-12b-it-qat-q4_0-unquantized',
        cache_dir=CACHE,
        allow_patterns=['*.json', '*.safetensors', '*.model', 'tokenizer*', 'special_tokens*'],
    )
    print('  ', gemma)
except GatedRepoError as e:
    raise SystemExit(
        '\n\nGemma access denied. Accept the license at '
        'https://huggingface.co/google/gemma-3-12b-it-qat-q4_0-unquantized '
        'and use a Read token from the same account.\n\n' + str(e)
    )

# Persist resolved paths for cell 6 (this small file goes in /kaggle/working).
with open('/kaggle/working/_paths.json', 'w') as f:
    json.dump({'ckpt': ckpt, 'gemma': gemma}, f)

print('\nRemaining /tmp:'); !df -h /tmp

## 6. Launch the UI with a public link

On the free T4 (16 GB VRAM) we run with `--quantization fp8-cast` and the UI's `one-stage` pipeline.

In [ ]:
import os, json
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['HF_HOME'] = '/tmp/hf_cache'
os.environ['HUGGINGFACE_HUB_CACHE'] = '/tmp/hf_cache'

with open('/kaggle/working/_paths.json') as f:
    paths = json.load(f)
ckpt, gemma = paths['ckpt'], paths['gemma']
print('Checkpoint:', ckpt)
print('Gemma:', gemma)

!HF_HOME=/tmp/hf_cache HUGGINGFACE_HUB_CACHE=/tmp/hf_cache \
  PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  python app.py \
    --checkpoint-path "{ckpt}" \
    --gemma-root "{gemma}" \
    --quantization fp8-cast \
    --output-dir /kaggle/working/outputs \
    --host 0.0.0.0 --port 7860 \
    --share

Look for `Running on public URL: https://*.gradio.live` and open that.

### Tips for free Kaggle

- Default pipeline is **`one-stage`** — keep it on the free T4.
- Width/height ≤ 512, frames ≤ 65.
- Generated MP4s land in `/kaggle/working/outputs` — visible in the right-side **Output** panel.
- Free Kaggle gives you 30 GPU-hours/week.
- ⚠️ If your session restarts, weights in `/tmp` are wiped — just re-run cells 4–6 (cell 3's pip install survives in /opt).

## (Optional) 7. Stop the server

In [ ]:
!pkill -f 'python app.py' || true